# Experiment 7: Gaussian Naive Bayes (manual and scikit-learn) for spam/ham

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score

# Spambase: 57 numeric features per email, class 1 = spam, 0 = ham
spam = fetch_openml(data_id=44, as_frame=False, parser="auto")
X = spam.data.astype(float)
y = spam.target.astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

In [ ]:
class ManualGaussianNB:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.mu_, self.var_, self.prior_ = [], [], []
        for c in self.classes_:
            Xc = X[y == c]
            self.mu_.append(Xc.mean(axis=0))
            self.var_.append(Xc.var(axis=0) + 1e-9)   # small value avoids division by zero
            self.prior_.append(len(Xc) / len(X))
        self.mu_, self.var_ = np.array(self.mu_), np.array(self.var_)
        return self

    def predict(self, X):
        log_post = []
        for k in range(len(self.classes_)):
            log_lik = -0.5 * np.sum(np.log(2 * np.pi * self.var_[k])
                                    + (X - self.mu_[k]) ** 2 / self.var_[k], axis=1)
            log_post.append(log_lik + np.log(self.prior_[k]))
        return self.classes_[np.argmax(np.array(log_post), axis=0)]

manual = ManualGaussianNB().fit(X_tr, y_tr)
p_manual = manual.predict(X_te)
print("Manual GNB   accuracy:", round(accuracy_score(y_te, p_manual), 3),
      " F1:", round(f1_score(y_te, p_manual), 3))

In [ ]:
sk = GaussianNB().fit(X_tr, y_tr)
p_sk = sk.predict(X_te)
print("sklearn GNB  accuracy:", round(accuracy_score(y_te, p_sk), 3),
      " F1:", round(f1_score(y_te, p_sk), 3))
print("Predictions agree on", round(np.mean(p_manual == p_sk) * 100, 1), "% of test emails")